# Session 3 — From Model to Clinic

*From Pixels to Patients · AAPM 2026 (Session 3 of 3)*

A trained model can't tell you when to distrust its output. Session 3 adds two QC tools:

1. **Reference-free QC** (`qc_workflow`) — score a contour with **no ground truth**, on two axes:
   **shape** (radiomic outlier?) and **position** (where a GTV sits?), plus **input validation**.
2. **Reference-based comparison** (`contour_compare`) — when a manual contour *does* exist, grade
   the AI against it with editing-effort metrics (surface Dice, added path length, time saved, 1–5 rating).

Cells 1–8 cover the reference-free check; cell 9 covers the comparison tools. No ground truth is
used in QC — the training *distribution* is the reference.

In [ ]:
# Dependencies (no GPU, no trained model needed — the check reasons about the data)
%pip install -q numpy scipy nibabel pandas matplotlib

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the Session 3 package importable whether you launch from the repo root or here.
for base in (Path.cwd(), *Path.cwd().parents):
    if (base / 'Session3' / 'qc_workflow').is_dir():
        sys.path.insert(0, str(base / 'Session3')); break
    if (base / 'qc_workflow').is_dir():
        sys.path.insert(0, str(base)); break

from qc_workflow import (QCConfig, discover_cases, find_nifti_root, split_by_patient,
                         characterize_cohort, build_reference, load_case, check_contour)
from run_qc_checks import _corrupt_variants
cfg = QCConfig()
cfg

## 1 · Find the Session 1 export and split by patient

Reuse the exact NIfTI tree Session 2 trained on. Split **by patient** so none contributes to
both the reference and the held-out set.

In [ ]:
nifti_root = find_nifti_root() or cfg.nifti_root
records = discover_cases(nifti_root, required_masks=(cfg.target_mask,))
print(f'{len(records)} cases under {nifti_root}')
assert len(records) >= 4, 'Run Session 1 export first (a few NSCLC-Radiomics patients).'
splits = split_by_patient(records, holdout_fraction=0.3, seed=cfg.seed)
print('reference:', len(splits['reference']), '| holdout:', len(splits['holdout']))

## 2 · Characterize the training data

The cohort *is* the reference. Distill it into acquisition ranges (spacing, HU) and the GTV's
shape + positional distributions — what we score against later.

In [ ]:
profile = characterize_cohort(splits['reference'], cfg)
print(profile.summary())

In [ ]:
# The GTV volume distribution is the simplest slice of the training profile.
vols = profile.feature_matrix[:, profile.feature_names.index('volume_cc')]
plt.figure(figsize=(5, 3))
plt.hist(vols, bins=12, color='#4C72B0', edgecolor='white')
plt.xlabel('GTV volume (cc)'); plt.ylabel('reference cases'); plt.title('Training GTV volume distribution')
plt.tight_layout(); plt.show()

## 3 · Build the reference-free check

Freeze the shape OOD model (robust Mahalanobis) and the position atlas (Procrustes constellation
of GTV + lungs/heart/cord/esophagus) into one portable object — what ships next to the model.

In [ ]:
reference = build_reference(profile, cfg)
reference.save(Path(cfg.output_dir) / 'reference_check.json')
print('atlas organs:', sorted(reference.atlas.get('template', {})))
print('shape features:', len(reference.shape['feature_names']))

## 4 · Validate inputs and check outputs

For each held-out GTV, score **input** (CT? plausible spacing?), **shape**, and **position**, and
take the worst band. A normal case should pass.

In [ ]:
def context_centroids(masks, spacing):
    ctx = {}
    for m in cfg.context_masks:
        arr = masks.get(m)
        if arr is not None and arr.sum() > 0:
            zz, yy, xx = np.where(arr > 0.5)
            ctx[m] = [float(zz.mean()*spacing[0]), float(yy.mean()*spacing[1]), float(xx.mean()*spacing[2])]
    return ctx

for rec in splits['holdout']:
    image, masks, spacing = load_case(rec, (cfg.target_mask, *cfg.context_masks))
    rep = check_contour(image, masks.get(cfg.target_mask), context_centroids(masks, spacing),
                        spacing, reference, cfg, profile.spacing)
    print(f"{rec.case_id[:36]:<36} {rep['verdict'].upper():<7} " + ('; '.join(rep['reasons']) or 'looks normal'))

## 5 · Watch it catch silent failures

Corrupt one good case three ways — **mislocated**, **over-segmented**, and **non-CT** — and
confirm each axis fires: position, shape, and input validation respectively.

In [ ]:
demo = splits['holdout'][0]
image, masks, spacing = load_case(demo, (cfg.target_mask, *cfg.context_masks))
ctx = context_centroids(masks, spacing)
for kind, (img, msk) in _corrupt_variants(image, masks[cfg.target_mask], spacing).items():
    rep = check_contour(img, msk, ctx, spacing, reference, cfg, profile.spacing)
    print(f"{kind:<16} {rep['verdict'].upper():<7} shape z={rep['shape'].get('z')}  pos z={rep['position'].get('z')}")
    print('   ->', '; '.join(rep['reasons']))

## 6 · Extend to the whole structure set (GTV + OARs)

Fit a **panel**: one shape reference per structure plus a single shared position atlas. OARs are
anatomically stereotyped, so their shape distributions are tight and outliers are meaningful.

In [ ]:
from qc_workflow import characterize_panel, build_panel, check_panel
prof = characterize_panel(splits['reference'], cfg)
panel = build_panel(prof, cfg)
panel.save(Path(cfg.output_dir) / 'panel_check.json')
for s in panel.structures:
    print(f'  {s:<10} {len(prof.feature_matrices[s]):>3} reference examples')

In [ ]:
# Score every structure on a held-out case
rec = splits['holdout'][0]
image, masks, spacing = load_case(rec, (cfg.target_mask, *cfg.context_masks))
for name, rep in check_panel(image, masks, spacing, panel, cfg, prof.spacing).items():
    print(f"  {name:<10} {rep['verdict'].upper():<7} " + ('; '.join(rep.get('reasons', [])) or 'ok'))

## 7 · Catch a swapped Lung_L / Lung_R

The classic labelling error. Swapping the two lung masks should flag **both** as wrong-side —
each lands where its partner belongs.

In [ ]:
swapped = dict(masks)
swapped['lung_l'], swapped['lung_r'] = masks['lung_r'], masks['lung_l']
res = check_panel(image, swapped, spacing, panel, cfg, prof.spacing)
for n in ('lung_l', 'lung_r'):
    print(f"  {n:<8} {res[n]['verdict'].upper():<7} {'; '.join(res[n]['reasons'])}")

## 8 · Package for governance

Write the training profile, frozen panel, per-structure report, and QC card — the artifacts a
monitoring process consumes. (`run_qc_checks.py` does all of this in one call.)

In [ ]:
import subprocess, sys as _sys
root = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / 'Session3').is_dir())
print(subprocess.run([_sys.executable, str(root / 'Session3' / 'run_qc_checks.py')],
                     capture_output=True, text=True).stdout)

## 9 · Reference-based comparison (`contour_compare`)

When a manual/corrected contour exists, grade the AI against it with the metrics that predict
**editing effort** — surface Dice @1/2/3 mm and added path length (APL) — plus a time-saving
estimate and a 1–5 clinical-acceptability rating. APL correlated R=0.87 with correction time
(Vaassen et al. 2020); volumetric Dice, which weights the untouched interior, did not.

In [ ]:
# Grade an AI contour against the manual one (reference-based)
from contour_compare import compare_masks
from compare_contours import synth_ai_contour

rec = splits['holdout'][0]
image, masks, spacing = load_case(rec, (cfg.target_mask,))
manual = masks[cfg.target_mask]
ai = synth_ai_contour(manual, grow_iters=2, shift_vox=1, region_frac=0.6)  # fake a realistic AI GTV
print(compare_masks(manual, ai, spacing, structure='gtv (AI vs manual)', rate_cm_min=3.0).summary())

In [ ]:
# Walk the 1-5 rubric by increasing edit severity
sweep = [('near-perfect', dict(grow_iters=1, shift_vox=0,  region_frac=0.15)),
         ('minor',        dict(grow_iters=1, shift_vox=1,  region_frac=0.30)),
         ('moderate',     dict(grow_iters=2, shift_vox=1,  region_frac=0.60)),
         ('poor',         dict(grow_iters=3, shift_vox=3,  region_frac=0.85)),
         ('unusable',     dict(grow_iters=8, shift_vox=12, region_frac=1.00))]
print(f"{'variant':<13}{'Dice':>6}{'sDSC2mm':>9}{'%saved':>8}{'rating':>8}")
for name, kw in sweep:
    r = compare_masks(manual, synth_ai_contour(manual, **kw), spacing, rate_cm_min=3.0)
    print(f"{name:<13}{r.dice:>6.2f}{r.surface_dice[2.0]:>9.2f}{r.timing.pct_saved:>7.0f}%{r.rating.score:>6}/5")
# Volumetric Dice stays high across the whole range; surface Dice + %saved track quality.

---
### Scope
Workshop artifacts, not clinical devices: **prioritizers for human review**, trained on **real CT
only** (MR / pseudo-CT rejected at input validation). On a small cohort, illustrative not reliable.

**Companion package:** `contour-qc`. **Prev:** Session 2 built the model this session guards.